# LP IV Practical 6 - Transfer Learning (Classification + Detection)
**Bird / Cat / Dog using MobileNetV2 and YOLO**

## Aim
Use a pre-trained network (transfer learning) to:
- **Part A** - classify images into 3 classes (bird, cat, dog) using MobileNetV2 + custom classifier + fine-tuning
- **Part B** - detect and draw boxes around bird, cat and dog using a pre-trained YOLO model fine-tuned on our dataset

## Flow
**Part A:** Set parameters -> Load data -> Load MobileNetV2 -> Freeze it -> Add classifier -> Compile -> Train -> Evaluate -> Fine-tune -> Final evaluation -> Plot graphs

**Part B:** Install ultralytics -> Load YOLO -> Train -> Validate -> Predict on a test image -> Display result

# PART A - Image Classification using MobileNetV2

## Block 1 - Import Libraries
- `tensorflow`: deep learning framework (includes Keras)
- `numpy`: array maths
- `matplotlib.pyplot`: plotting accuracy/loss graphs
- `ImageDataGenerator`: loads images from folders, rescales them and creates augmented copies
- `MobileNetV2`: a light CNN already trained on ImageNet
- `GlobalAveragePooling2D`: shrinks each feature map to a single number
- `Dense`: fully connected layer
- `Dropout`: randomly switches off neurons to reduce overfitting
- `Model`: joins an input and an output into one model
- `Adam`: optimizer that updates the weights

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

## Block 2 - Set Parameters and Dataset Paths
- `IMG_SIZE = (128, 128)`: every image is resized to 128 x 128 so all inputs have the same shape
- `BATCH_SIZE = 32`: number of images processed at one time
- Folder names are the class names:
```
dataset/
  train/        bird/  cat/  dog/
  validation/   bird/  cat/  dog/
  test/         bird/  cat/  dog/
```
- The `r"..."` raw string stops Windows backslashes being treated as special characters.

**Change `BASE_PATH` to your own dataset location.**

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

BASE_PATH = r'C:\Users\Sairam\OneDrive\Desktop\IT\dataset'

TRAIN_PATH = BASE_PATH + r'\train'
VALIDATION_PATH = BASE_PATH + r'\validation'
TEST_PATH = BASE_PATH + r'\test'

## Block 3 - Data Preprocessing (Generators)
- `rescale = 1/255` converts pixel values from 0-255 to 0-1 so training is faster and more stable.
- Training data also gets **augmentation**: rotation up to 20 degrees, 20% horizontal/vertical shift, random left-right flip. This creates many versions of each image, which helps with small datasets and reduces overfitting.
- Validation and test data are only rescaled. They must **not** be augmented because they measure real performance.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

validation_datagen = ImageDataGenerator(rescale=1.0 / 255)

test_datagen = ImageDataGenerator(rescale=1.0 / 255)

## Block 4 - Load Train, Validation and Test Data
- `flow_from_directory` treats each sub-folder as one class.
- `class_mode='categorical'` creates one-hot labels (e.g. cat = [0, 1, 0]).
- `shuffle=True` for training mixes the order every epoch.
- `shuffle=False` for validation/test keeps a fixed order so results can be matched with labels.

Output in the original run: 3 train, 3 validation, 2 test images (a **very** small dataset).

In [ ]:
train_data = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

validation_data = validation_datagen.flow_from_directory(
    VALIDATION_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_data = test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

## Block 5 - Number of Classes
The final layer needs to know how many outputs to create. Classes are numbered alphabetically by folder name: `{'bird': 0, 'cat': 1, 'dog': 2}`.

In [ ]:
num_classes = train_data.num_classes

print('\nNumber of classes:', num_classes)
print('Class names:', train_data.class_indices)

## Block 6 - Load Pre-trained MobileNetV2
- `weights='imagenet'`: uses weights already learned from millions of ImageNet images (edges, textures, shapes, fur, feathers...).
- `include_top=False`: removes the original 1000-class output layers so we can attach our own 3-class classifier.
- `input_shape=(128,128,3)`: 128 x 128 colour (RGB) images.

This is the core idea of **transfer learning**: reuse knowledge from a big task for a new, smaller task.

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128, 128, 3)
)

## Block 7 - Freeze the Pre-trained Layers
In the first phase only our new layers should learn. If the base layers were trained now, the random new layers would send large wrong updates backwards and damage the useful pre-trained features.

In [ ]:
base_model.trainable = False

## Block 8 - Add the Custom Classifier
- `GlobalAveragePooling2D`: converts the (4, 4, 1280) feature maps into one vector of 1280 values
- `Dense(128, relu)`: learns to combine the features for our 3 classes
- `Dropout(0.5)`: randomly drops 50% of neurons during training to reduce overfitting
- `Dense(num_classes, softmax)`: 3 outputs (bird, cat, dog) as probabilities that sum to 1

In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(num_classes, activation='softmax')(x)

## Block 9 - Create the Final Model
Joins MobileNetV2 (input) and our classifier (output) into a single Keras `Model` that can be compiled, trained and used for prediction.

In [ ]:
model = Model(inputs=base_model.input, outputs=output)

## Block 10 - Compile the Model and View Summary
- **Adam (lr = 0.001)**: adjusts the weights to reduce the loss and adapts the step size automatically
- **categorical_crossentropy**: correct loss for multi-class classification with one-hot labels
- **accuracy**: percentage of correct predictions

Original run: total params 2,422,339; trainable 164,355 (only our new layers); non-trainable 2,257,984 (frozen MobileNetV2).

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Block 11 - Train the Custom Classifier (Phase 1)
Trains only the new top layers for 5 epochs (one epoch = the model sees the full training set once). `validation_data` checks performance after every epoch. `history1` stores accuracy/loss per epoch for plotting.

In [ ]:
history1 = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=5
)

## Block 12 - Evaluate Before Fine-Tuning
Tests the model on the unseen test set to get a baseline. Original run: test loss 1.2990, test accuracy 0.5 (with only 2 test images, 50% means 1 image was correct).

In [ ]:
test_loss1, test_accuracy1 = model.evaluate(test_data)

print('\nTest Loss Before Fine-Tuning:', test_loss1)
print('Test Accuracy Before Fine-Tuning:', test_accuracy1)

## Block 13 - Fine-Tuning: Unfreeze the Last 20 Layers
- Early layers learn general features (edges, colours) useful for any image task, so they stay frozen.
- Later layers learn task-specific features, so retraining them lets the model adapt to birds, cats and dogs.
- `base_model.trainable = True` unfreezes everything first; the loop then freezes all layers except the last 20.

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

## Block 14 - Recompile with a Small Learning Rate
- Compiling must be repeated after changing which layers are trainable, otherwise the change does not take effect.
- The learning rate is 10x smaller (0.0001) so the pre-trained weights are only adjusted gently and not destroyed.

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## Block 15 - Fine-Tune the Model (Phase 2)
Trains 5 more epochs with the last 20 base layers unfrozen. `history2` stores this phase separately so both phases can be joined into one graph afterwards.

In [ ]:
history2 = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=5
)

## Block 16 - Final Evaluation
Tests the fine-tuned model on the test set; compare with Block 12. Original run: loss improved from 1.2990 to 0.8183, accuracy stayed 0.5 (the test set has only 2 images).

In [ ]:
test_loss2, test_accuracy2 = model.evaluate(test_data)

print('\nFinal Test Loss:', test_loss2)
print('Final Test Accuracy:', test_accuracy2)

## Block 17 - Combine Training History
`history1` and `history2` are 5 epochs each. Adding the lists gives one continuous record of 10 epochs for plotting.

In [ ]:
accuracy = history1.history['accuracy'] + history2.history['accuracy']
val_accuracy = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']

## Block 18 - Plot Accuracy
Training vs validation accuracy over 10 epochs. The dashed vertical line marks where fine-tuning starts. If training accuracy rises while validation accuracy stays low, the model is overfitting. Curves are jumpy here because the dataset is tiny.

*Tip: `x=4.5` places the line exactly between the two phases.*

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(accuracy, label='Training Accuracy')
plt.plot(val_accuracy, label='Validation Accuracy')
plt.axvline(x=4, linestyle='--', label='Fine-Tuning Starts')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('MobileNetV2 Transfer Learning Accuracy')
plt.legend()
plt.show()

## Block 19 - Plot Loss
Loss should go down as the model learns. With only 3 training images it is not a reliable measure.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.axvline(x=4, linestyle='--', label='Fine-Tuning Starts')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('MobileNetV2 Transfer Learning Loss')
plt.legend()
plt.show()

# PART B - Object Detection using YOLO

## Block 20 - Install the Ultralytics Library
Fixes `ModuleNotFoundError: No module named 'ultralytics'`. Run the cell below once, then **restart the kernel** and continue. The first run also downloads `yolo11n.pt` (internet needed).

In [ ]:
!pip install ultralytics opencv-python

## Block 21 - Import Libraries and Load Pre-trained YOLO
- YOLO ("You Only Look Once") finds objects and draws bounding boxes in a single pass, so it is fast.
- `yolo11n.pt` is the "nano" version: smallest and fastest, trained on COCO (which already includes bird, cat and dog).
- `cv2` converts image colours; `matplotlib` displays the image.
- The variable is named `yolo_model` so it does not overwrite the Part A `model`.

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2

yolo_model = YOLO('yolo11n.pt')

## Block 22 - Dataset Format Required by YOLO (`data.yaml`)
YOLO needs **bounding-box labels**, not just class folders. Each image needs a `.txt` file with lines like:

`class_id  x_center  y_center  width  height`  (values between 0 and 1)

Folder layout:
```
dataset/
  images/train  images/val  images/test
  labels/train  labels/val  labels/test
```
Create `data.yaml` in the dataset folder (the cell below writes it for you; edit the path if needed):
```yaml
path: C:/Users/Sairam/OneDrive/Desktop/IT/dataset
train: images/train
val: images/val
test: images/test
names:
  0: bird
  1: cat
  2: dog
```
Labels can be created with tools like LabelImg, CVAT or Roboflow. The plain class folders used in Part A are **not** enough for this step.

In [ ]:
yaml_text = """path: C:/Users/Sairam/OneDrive/Desktop/IT/dataset
train: images/train
val: images/val
test: images/test
names:
  0: bird
  1: cat
  2: dog
"""

with open(BASE_PATH + r'\data.yaml', 'w') as f:
    f.write(yaml_text)

print('data.yaml saved')

## Block 23 - Train / Fine-Tune YOLO
- `data`: path to `data.yaml`
- `epochs`: 10 passes over the training data
- `imgsz`: images resized to 640 x 640
- `batch`: 8 images per update (small value saves memory)
- `name`: results are saved in `runs/detect/bird_cat_dog_detection`

This is transfer learning again: YOLO already knows how to detect objects, and we adapt it to our data.

In [ ]:
results = yolo_model.train(
    data=BASE_PATH + r'\data.yaml',
    epochs=10,
    imgsz=640,
    batch=8,
    name='bird_cat_dog_detection'
)

## Block 24 - Validate the Model
Measures detection quality on the validation set (precision, recall, mAP50, mAP50-95). mAP (mean Average Precision) is the standard detection metric: higher is better (max 1.0).

In [ ]:
metrics = yolo_model.val()

print('\nValidation completed')
print('mAP50    :', metrics.box.map50)
print('mAP50-95 :', metrics.box.map)

## Block 25 - Predict on a Test Image
- `image_path`: location of the picture to test
- `conf=0.25`: detections below 25% confidence are ignored (raise to 0.5 for fewer but more certain detections)

In [ ]:
image_path = r'C:\Users\Sairam\OneDrive\Desktop\IT\dataset\test.jpg'

results = yolo_model.predict(
    source=image_path,
    conf=0.25
)

## Block 26 - Display the Detection Result
- `result.plot()` returns the image with boxes, class names and confidence scores drawn on it.
- OpenCV uses BGR colour order while matplotlib uses RGB, so `cv2.cvtColor(..., COLOR_BGR2RGB)` fixes the colours.
- `plt.axis('off')` hides the axis numbers.

In [ ]:
for result in results:
    annotated_image = result.plot()
    annotated_image = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 8))
    plt.imshow(annotated_image)
    plt.axis('off')
    plt.title('Bird, Cat and Dog Object Detection')
    plt.show()

## Block 27 - Conclusion
An f-string always shows the current computed numbers, even if the model is retrained.

In [ ]:
print(f'MobileNetV2 transfer learning: Test Loss = {test_loss2:.4f}, '
      f'Test Accuracy = {test_accuracy2:.4f}')
print('YOLO fine-tuned model detects birds, cats and dogs in images.')

## Notes
1. **Dataset too small:** only 3 train, 3 validation and 2 test images, so 50% test accuracy means 1 of 2 correct. Use at least 50-100 images per class.
2. **YOLO error:** caused by the missing `ultralytics` library (Block 20).
3. **YOLO data:** needs bounding-box labels and a `data.yaml` file (Block 22).
4. **Variable name:** `yolo_model` is used so the Part A `model` is not overwritten.
5. Results vary slightly run to run because of random augmentation and weight initialization.

**Libraries:** `pip install tensorflow numpy matplotlib ultralytics opencv-python`